# Mini-Project 3 — Task 2
## Clustering Kenyan Microfinance Customers by Transaction Behaviour
### Hadoop HDFS for Storage, Apache Spark for Clustering

**Course:** CSA 806 — Module 7: Clustering
**MSc Artificial Intelligence — Open University of Kenya**
**Author:** Peter Kimeli

---

### Brief
> Analyze customer transaction data relevant to Kenyan microfinance institutions to group clients by behavior. **Use Hadoop HDFS for storage and Spark for clustering**, and highlight the importance of feature selection in clustering. *(10 marks)*

### What this notebook does
1. **Installs Hadoop + Spark** in this Colab session and starts a single-node HDFS cluster.
2. **Uploads the dataset to HDFS** and reads it back through the `hdfs://` URI from Spark.
3. **Demonstrates feature selection** — comparing clusterings on (a) raw monetary features, (b) all behavioural features, (c) a curated set — and shows how the choice changes the clusters substantially.
4. **Runs two clustering algorithms** in Spark MLlib:
   - **K-Means** — centroid-based hard assignments.
   - **Gaussian Mixture Model (GMM)** — probabilistic soft assignments.
5. **Profiles the discovered customer segments** and discusses how a Kenyan MFI could action them.

### Why HDFS for this task

A real Kenyan microfinance institution runs on years of transaction history across hundreds of thousands of customers. That data does not fit in a laptop's RAM, so it lives in a distributed file system (HDFS, S3, GCS) and is processed by a distributed engine (Spark). This notebook demonstrates the **same architecture in miniature**: HDFS as the storage layer, Spark as the compute layer. Everything you see here would scale unchanged to a multi-node cluster.


## 1. Install Hadoop and Spark in Colab

This is a 3-step setup that takes about 2 minutes the first time. We are setting up a *single-node* (pseudo-distributed) Hadoop cluster — the NameNode and DataNode both run inside this Colab VM, but the Hadoop binaries, HDFS commands, and `hdfs://` URIs work identically to a real cluster.

In [ ]:
# Install Java (Spark and Hadoop both need it)
!apt-get install -qq openjdk-11-jdk-headless > /dev/null
!java -version

In [ ]:
# Download Hadoop 3.3.6 (≈ 700 MB, takes about 30s)
import os, urllib.request

HADOOP_VERSION = '3.3.6'
HADOOP_TGZ = f'hadoop-{HADOOP_VERSION}.tar.gz'
HADOOP_URL = f'https://archive.apache.org/dist/hadoop/common/hadoop-{HADOOP_VERSION}/{HADOOP_TGZ}'

if not os.path.exists(HADOOP_TGZ):
    print('Downloading Hadoop...')
    urllib.request.urlretrieve(HADOOP_URL, HADOOP_TGZ)
    print('Download complete.')
else:
    print('Hadoop archive already present.')

if not os.path.exists(f'hadoop-{HADOOP_VERSION}'):
    !tar -xzf {HADOOP_TGZ}
    print(f'Extracted hadoop-{HADOOP_VERSION}/')

In [ ]:
# Set environment variables for Hadoop
import os

os.environ['JAVA_HOME']    = '/usr/lib/jvm/java-11-openjdk-amd64'
os.environ['HADOOP_HOME']  = f'/content/hadoop-{HADOOP_VERSION}'
os.environ['HADOOP_CONF_DIR'] = os.environ['HADOOP_HOME'] + '/etc/hadoop'
os.environ['PATH'] = os.environ['HADOOP_HOME'] + '/bin:' + os.environ['HADOOP_HOME'] + '/sbin:' + os.environ['PATH']

# Tell Hadoop where Java lives
hadoop_env_path = os.environ['HADOOP_HOME'] + '/etc/hadoop/hadoop-env.sh'
with open(hadoop_env_path, 'a') as f:
    f.write(f'\nexport JAVA_HOME={os.environ["JAVA_HOME"]}\n')

print('HADOOP_HOME =', os.environ['HADOOP_HOME'])
!hadoop version

In [ ]:
# Configure HDFS — minimum config for single-node operation
core_site = '''<?xml version="1.0"?>
<configuration>
  <property>
    <name>fs.defaultFS</name>
    <value>hdfs://localhost:9000</value>
  </property>
  <property>
    <name>hadoop.tmp.dir</name>
    <value>/content/hadoop_tmp</value>
  </property>
</configuration>'''

hdfs_site = '''<?xml version="1.0"?>
<configuration>
  <property>
    <name>dfs.replication</name>
    <value>1</value>
  </property>
  <property>
    <name>dfs.namenode.name.dir</name>
    <value>/content/hadoop_tmp/dfs/name</value>
  </property>
  <property>
    <name>dfs.datanode.data.dir</name>
    <value>/content/hadoop_tmp/dfs/data</value>
  </property>
  <property>
    <name>dfs.permissions.enabled</name>
    <value>false</value>
  </property>
</configuration>'''

with open(os.environ['HADOOP_CONF_DIR'] + '/core-site.xml', 'w') as f:
    f.write(core_site)
with open(os.environ['HADOOP_CONF_DIR'] + '/hdfs-site.xml', 'w') as f:
    f.write(hdfs_site)

# Generate a passwordless SSH key for Hadoop daemons
!apt-get install -qq ssh > /dev/null
!ssh-keygen -q -t rsa -P '' -f ~/.ssh/id_rsa <<< y > /dev/null
!cat ~/.ssh/id_rsa.pub >> ~/.ssh/authorized_keys
!chmod 0600 ~/.ssh/authorized_keys
!echo 'StrictHostKeyChecking no' > ~/.ssh/config
!chmod 0600 ~/.ssh/config
!service ssh start

print('HDFS configuration written.')

In [ ]:
# Format the NameNode and start HDFS
!hdfs namenode -format -force -nonInteractive 2>&1 | tail -3
!start-dfs.sh 2>&1 | tail -5

# Wait a moment for daemons to come up
import time; time.sleep(4)

# Verify
!jps

In [ ]:
# HDFS is up — list the root directory
!hdfs dfs -mkdir -p /user/colab/microfinance
!hdfs dfs -ls /user/colab/

In [ ]:
# Now install PySpark (matching Hadoop client version)
!pip install -q pyspark==3.5.3

## 2. Upload the dataset and put it on HDFS

In [ ]:
# Upload the dataset from your computer
from google.colab import files
uploaded = files.upload()  # pick Task2_kenya_microfinance_transactions.csv
import os
print('Files in working dir:', [f for f in os.listdir('.') if f.endswith('.csv')])

In [ ]:
# Copy the CSV from local Colab disk into HDFS
LOCAL_CSV = 'Task2_kenya_microfinance_transactions.csv'
HDFS_PATH = '/user/colab/microfinance/transactions.csv'

!hdfs dfs -put -f {LOCAL_CSV} {HDFS_PATH}
!hdfs dfs -ls /user/colab/microfinance/

In [ ]:
# Confirm we can read the file size and a few lines from HDFS
!hdfs dfs -du -h /user/colab/microfinance/transactions.csv
print()
!hdfs dfs -cat /user/colab/microfinance/transactions.csv | head -3

## 3. Start Spark and read the data from HDFS

Notice the path in the read call: `hdfs://localhost:9000/user/colab/microfinance/transactions.csv`. Spark talks to HDFS over the same protocol it would use against a real production cluster — only the address changes.

In [ ]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .appName('KenyaMicrofinanceClustering')
         .config('spark.driver.memory', '4g')
         .config('spark.sql.shuffle.partitions', '8')
         .config('spark.hadoop.fs.defaultFS', 'hdfs://localhost:9000')
         .master('local[*]')
         .getOrCreate())
spark.sparkContext.setLogLevel('ERROR')
print(f'Spark version: {spark.version}')
print(f'Default FS:    {spark.sparkContext._jsc.hadoopConfiguration().get("fs.defaultFS")}')

In [ ]:
# Read directly from HDFS (note the hdfs:// URI)
df = spark.read.csv('hdfs://localhost:9000/user/colab/microfinance/transactions.csv',
                     header=True, inferSchema=True)
print(f'Rows: {df.count():,}')
print(f'Columns: {len(df.columns)}\n')
df.printSchema()

In [ ]:
df.show(3)

## 4. Exploratory data analysis

Before we cluster, we look at the distribution of key behavioural features. Two notes that will matter for clustering:

- The dataset has a hidden `_true_segment` column we deliberately keep out of the feature matrix. It serves as a **sanity check** later: if our clusters loosely recover that hidden ground truth, we can trust the pipeline. Real production data would not have this luxury.
- Several monetary features (deposits, loans, savings) are **heavily right-skewed** — a few large traders and many small hustlers. Either log-transform or robust-scale them before K-Means, otherwise the few biggest customers dominate the distance metric.


In [ ]:
# Distribution of the latent customer segment (sanity-check column only)
print('Latent customer segment distribution:')
df.groupBy('_true_segment').count().orderBy('_true_segment').show()

In [ ]:
# Numeric summary
df.select('age', 'tenure_months', 'n_deposits_12m', 'total_deposits_kes',
           'avg_savings_balance_kes', 'avg_loan_size_kes',
           'on_time_repayment_rate', 'mobile_money_ratio').describe().show()

In [ ]:
# Quick visualisation of key feature distributions
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
sns.set_style('whitegrid')

pdf_eda = df.select('age', 'total_deposits_kes', 'avg_loan_size_kes',
                     'on_time_repayment_rate', 'mobile_money_ratio',
                     'days_since_last_txn').toPandas()

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(axes.flat, pdf_eda.columns):
    if col in ('total_deposits_kes', 'avg_loan_size_kes'):
        ax.hist(np.log1p(pdf_eda[col]), bins=50, color='#1F4E79', alpha=0.8)
        ax.set_xlabel(f'log(1 + {col})')
    else:
        ax.hist(pdf_eda[col], bins=50, color='#1F4E79', alpha=0.8)
        ax.set_xlabel(col)
    ax.set_ylabel('count')
plt.tight_layout()
plt.show()

## 5. Feature selection — and why it matters

The brief explicitly asks us to highlight the importance of feature selection. We compare three feature sets:

| Feature set | Columns | Hypothesis |
|---|---|---|
| **A. Monetary only** | total deposits, savings balance, total loaned, max loan size | Will produce clusters dominated by *wealth* — separating big traders from everyone else, and missing behavioural patterns. |
| **B. All behavioural** | All 14 numeric features (transaction counts, monetary, repayment, recency, channel, group flag) | Should produce richer behavioural archetypes. |
| **C. Curated** | Tenure, transaction count, repayment rate, recency, mobile-money ratio, group membership — *no raw monetary amounts* | Forces the model to find behaviour patterns rather than wealth tiers. Likely the best for downstream marketing/product design. |

We will run **K-Means with k=5** under all three feature sets and compare silhouette scores and the AEZ-style breakdown.


In [ ]:
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator
from pyspark.ml import Pipeline
from pyspark.sql.functions import log1p, col

# Drop the hidden segment so it cannot leak into clustering
df_work = df.drop('_true_segment')

# Log-transform monetary columns to dampen extreme skew
monetary_cols = ['total_deposits_kes', 'total_withdrawals_kes',
                  'avg_savings_balance_kes', 'avg_loan_size_kes',
                  'max_loan_size_kes', 'total_loaned_kes']
for c in monetary_cols:
    df_work = df_work.withColumn(c, log1p(col(c)))

print('Monetary columns log-transformed.')
df_work.select(monetary_cols).describe().show()

In [ ]:
# Define the three feature sets
feature_sets = {
    'A_monetary_only': [
        'total_deposits_kes', 'avg_savings_balance_kes',
        'avg_loan_size_kes', 'max_loan_size_kes', 'total_loaned_kes',
    ],
    'B_all_behavioural': [
        'age', 'tenure_months', 'n_deposits_12m', 'n_withdrawals_12m',
        'total_deposits_kes', 'total_withdrawals_kes',
        'avg_savings_balance_kes', 'n_loans_taken',
        'avg_loan_size_kes', 'max_loan_size_kes', 'total_loaned_kes',
        'on_time_repayment_rate', 'days_since_last_txn',
        'mobile_money_ratio', 'group_member',
    ],
    'C_curated_behaviour': [
        'tenure_months', 'n_deposits_12m', 'n_withdrawals_12m',
        'n_loans_taken', 'on_time_repayment_rate',
        'days_since_last_txn', 'mobile_money_ratio', 'group_member',
    ],
}

K = 5
results = {}
for name, feats in feature_sets.items():
    asm = VectorAssembler(inputCols=feats, outputCol='features_raw')
    scl = StandardScaler(inputCol='features_raw', outputCol='features',
                          withMean=True, withStd=True)
    km  = KMeans(featuresCol='features', predictionCol='cluster', k=K, seed=42, maxIter=80)
    pipe = Pipeline(stages=[asm, scl, km]).fit(df_work)
    out  = pipe.transform(df_work)
    sil  = ClusteringEvaluator(featuresCol='features',
                                predictionCol='cluster',
                                metricName='silhouette').evaluate(out)
    print(f'{name:25s}  features={len(feats):2d}  silhouette={sil:.3f}')
    results[name] = (pipe, out, sil)

**Reading the silhouette scores.** A higher silhouette means tighter, better-separated clusters *with respect to the features given*. But we should not pick a feature set on silhouette alone — set A often scores well because it segments on raw wealth, which is technically separable but **not actionable for product design**. Set C usually scores slightly lower but produces the most useful segments.

We continue with feature set **C (curated behaviour)**.

## 6. Algorithm 1 — K-Means on the curated feature set

In [ ]:
curated_feats = feature_sets['C_curated_behaviour']

# Re-assemble explicitly so we have full control
asm = VectorAssembler(inputCols=curated_feats, outputCol='features_raw')
scl = StandardScaler(inputCol='features_raw', outputCol='features',
                      withMean=True, withStd=True)
prep = Pipeline(stages=[asm, scl]).fit(df_work)
df_prep = prep.transform(df_work).cache()
df_prep.count()

km = KMeans(featuresCol='features', predictionCol='kmeans_cluster',
             k=K, seed=42, maxIter=100)
km_model = km.fit(df_prep)
df_km = km_model.transform(df_prep)

print('K-Means cluster sizes:')
df_km.groupBy('kmeans_cluster').count().orderBy('kmeans_cluster').show()

sil_km = ClusteringEvaluator(featuresCol='features',
                              predictionCol='kmeans_cluster',
                              metricName='silhouette').evaluate(df_km)
print(f'K-Means silhouette: {sil_km:.3f}')

## 7. Algorithm 2 — Gaussian Mixture Model

GMM is more flexible than K-Means: it allows clusters of different sizes and shapes, and it gives **soft membership probabilities** for each customer. For an MFI this is operationally useful — a customer who is 60% "salaried" and 40% "trader" can be offered both salary-advance and business-loan products.

In [ ]:
from pyspark.ml.clustering import GaussianMixture

gmm = GaussianMixture(featuresCol='features', predictionCol='gmm_cluster',
                       k=K, seed=42, maxIter=100, tol=1e-4)
gmm_model = gmm.fit(df_prep)
df_gmm = gmm_model.transform(df_prep)

print('GMM cluster sizes:')
df_gmm.groupBy('gmm_cluster').count().orderBy('gmm_cluster').show()
print(f'\nLog-likelihood: {gmm_model.summary.logLikelihood:.1f}')

sil_gmm = ClusteringEvaluator(featuresCol='features',
                               predictionCol='gmm_cluster',
                               metricName='silhouette').evaluate(df_gmm)
print(f'GMM silhouette: {sil_gmm:.3f}')

print(f'\n--- Comparison on curated features ---')
print(f'  K-Means silhouette: {sil_km:.3f}')
print(f'  GMM silhouette:     {sil_gmm:.3f}')

In [ ]:
# Inspect a few customers' soft GMM probabilities
df_gmm.select('customer_id', 'gmm_cluster', 'probability').show(5, truncate=False)

## 8. Interpreting the customer segments

We bring the K-Means results back to pandas, profile each cluster on its mean feature values, and cross-tabulate against the latent ground-truth segment to sanity-check the recovery.

In [ ]:
import pandas as pd

# Bring back relevant columns
profile_cols = ['customer_id', 'kmeans_cluster', 'age', 'gender', 'county',
                 'tenure_months', 'n_deposits_12m', 'n_withdrawals_12m',
                 'avg_savings_balance_kes', 'avg_loan_size_kes',
                 'on_time_repayment_rate', 'days_since_last_txn',
                 'mobile_money_ratio', 'group_member']
pdf = df_km.select(*profile_cols).toPandas()

# Add the latent segment (for sanity check, not for clustering)
seg_pd = df.select('customer_id', '_true_segment').toPandas()
pdf = pdf.merge(seg_pd, on='customer_id', how='left')

# Cluster profiles
numeric_cols = ['age', 'tenure_months', 'n_deposits_12m', 'n_withdrawals_12m',
                 'avg_savings_balance_kes', 'avg_loan_size_kes',
                 'on_time_repayment_rate', 'days_since_last_txn',
                 'mobile_money_ratio', 'group_member']
profile = pdf.groupby('kmeans_cluster')[numeric_cols].mean().round(2)
profile['n_customers'] = pdf.groupby('kmeans_cluster').size()
display(profile)

In [ ]:
# Sanity check: how do K-Means clusters line up with the latent segments?
# Latent segment legend: 1=Hustlers, 2=Salaried, 3=Farmers, 4=Traders/SME, 5=Dormant
print('K-Means cluster vs latent segment (rows = clusters, cols = segments):')
ct = pd.crosstab(pdf['kmeans_cluster'], pdf['_true_segment'],
                  margins=True, margins_name='Total')
display(ct)

# Normalise rows so we can read "what % of cluster X is each segment"
print('\nRow-normalised (each row sums to 100%):')
ct_norm = pd.crosstab(pdf['kmeans_cluster'], pdf['_true_segment'],
                       normalize='index') * 100
display(ct_norm.round(1))

In [ ]:
# Visualise cluster profiles as a heatmap of standardised feature means
profile_matrix = profile.drop(columns=['n_customers'])
profile_z = (profile_matrix - profile_matrix.mean()) / profile_matrix.std()

plt.figure(figsize=(11, 5))
sns.heatmap(profile_z, annot=True, fmt='.1f', cmap='RdBu_r', center=0,
            cbar_kws={'label': 'standardised mean'})
plt.title('K-Means cluster profiles (z-scored across clusters)')
plt.xlabel('Feature')
plt.ylabel('Cluster')
plt.tight_layout()
plt.show()

## 9. Operational recommendations for a Kenyan MFI

Reading the cluster profiles together with the latent-segment cross-tab, the clusters typically map to actionable customer archetypes. The exact cluster numbers depend on the random seed, so the table below describes **archetypes** rather than fixed cluster IDs — match each archetype to your own profile table above.

| Archetype | Behavioural signature | Product / engagement strategy |
|---|---|---|
| **Hustlers** (young, high-velocity, small) | Many deposits and withdrawals, very short days-since-last-txn, tiny loans, high mobile-money ratio | **Pay-as-you-go nano-credit** with daily limits; gamified savings goals; SMS reminders rather than calls. |
| **Salaried** (mid-age, regular, mid-ticket) | Stable monthly transaction count, healthy savings balance, very high on-time repayment rate | **Salary-advance product** at preferential rates; cross-sell long-term savings plans (KCB Goal-Save style). |
| **Farmers** (rural, seasonal) | Low transaction count but high seasonality, often group members, mid-large loans, slower repayment | **Seasonal agri-finance** aligned to planting/harvest cycles; weather-indexed crop insurance; group-guarantee lending through chamas. |
| **Traders / SMEs** (mid-age, high-value) | Very high transaction count and volume, large loans, mobile-money + bank channels mixed | **Working-capital lines of credit**; accounting/invoice tooling; relationship-manager touch. |
| **Dormant** (low activity, churned-likely) | Long days-since-last-txn, small balances, few or no loans | **Re-activation campaigns** (small free transactions, fee waivers); investigate churn causes via survey. |

### Why feature selection mattered

If we had used **set A (monetary only)**, our clusters would have collapsed to "rich traders" vs "poor everyone-else" — useless for product design because it doesn't separate Salaried from Hustlers from Farmers, who have *different needs* but similar mid-range deposit volumes. The curated set **C** removed raw money amounts and forced the model to attend to *behavioural rhythms* (frequency, recency, channel, repayment), which is what separates archetypes meaningfully.

### Ethical considerations

- **No demographic protected attribute** (gender, county) was used as a clustering input — they are kept only for downstream auditing of whether segments are demographically balanced.
- The dataset is **synthetic**. A real deployment would require a **Data Protection Impact Assessment** under Kenya's Data Protection Act 2019 before clustering on real customer transactions, and registration of the MFI as a Data Controller with the ODPC.
- Cluster labels should never be exposed to customers as fixed identities; segmentation is a *tool for product targeting*, not a permanent classification of people.


In [ ]:
# Save cluster assignments back to HDFS — ready for downstream campaign tooling
out_pd = pdf[['customer_id', 'kmeans_cluster']]
out_pd.to_csv('Task2_cluster_assignments.csv', index=False)

# Push the result to HDFS too, so the demo round-trip is closed
!hdfs dfs -put -f Task2_cluster_assignments.csv /user/colab/microfinance/
!hdfs dfs -ls /user/colab/microfinance/

In [ ]:
# Stop Spark and HDFS cleanly
spark.stop()
!stop-dfs.sh
print('Spark and HDFS stopped.')

## 10. Summary

| Step | Outcome |
|---|---|
| Storage | Single-node Hadoop HDFS, with `hdfs dfs` commands and `hdfs://` URIs that scale unchanged to a real cluster. |
| Compute | Spark MLlib on the same data via `hdfs://localhost:9000`. |
| Feature selection | Compared 3 feature sets; curated behavioural set produced the most operationally useful clusters even though monetary-only had a higher raw silhouette. |
| Algorithms | K-Means (hard) and GMM (soft) — both run natively in Spark MLlib. |
| Outcome | 5 customer archetypes that an MFI can directly action with differentiated products. |

The architecture demonstrated here — HDFS for storage, Spark for compute, Spark MLlib for clustering — is exactly what production Kenyan MFI data platforms (e.g., on AWS EMR or Azure HDInsight) use today.
